# Env

In [ ]:
import os
import json
import glob

from tqdm import tqdm, trange
import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn.functional as F

from datasets import load_dataset
from transformers import (AutoTokenizer,
                          AutoModelForCausalLM,
                          BitsAndBytesConfig,
                          pipeline,
                          GenerationConfig,
                          TrainingArguments)
from peft import (LoraConfig,
                  PeftModel)
from trl import SFTTrainer

from sentence_transformers import SentenceTransformer

In [ ]:
# work dir
work_dir = '/home/ubuntu/nlp-practice'

In [ ]:
%cd {work_dir}
!pwd

In [ ]:
# tokeinzer warning disable
os.environ["TOKENIZERS_PARALLELISM"] = "false"

In [ ]:
MODEL_ID = "google/gemma-3-1b-it"
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# LLM을 이용한 문장분류 (NSMC)

## Gemma understanding

In [ ]:
# declare 4 bits quantize
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16
)
# load 4 bits model
model = AutoModelForCausalLM.from_pretrained(MODEL_ID,
                                             attn_implementation='eager',
                                             device_map='auto',
                                             quantization_config=quantization_config)
# load tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.padding_side = 'right'

gemma prompt 형식 확인
```
<bos><start_of_turn>user
{content}<end_of_turn>
<start_of_turn>model
```

In [ ]:
doc = """엄청나게 재밌습니다. 강추!!!"""

messages = [
    {
        "role": "user",
        "content": "다음 문장은 영화리뷰입니다. 긍정 또는 부정으로 분류해주세요:\n\n{}".format(doc)
    }
]
prompt = tokenizer.apply_chat_template(messages,
                                       tokenize=False,
                                       add_generation_prompt=True)
print(prompt)

In [ ]:
# generate 함수를 이용한 생성
x = tokenizer(
        prompt,
        truncation=True,
        max_length=512,
        return_tensors="pt",
    )["input_ids"].to(device)

beam_output = model.generate(
        input_ids=x
    )

result = tokenizer.decode(beam_output[0], skip_special_tokens=False)
print(result)

In [ ]:
# 파이프라인 선언
pipe = pipeline("text-generation",
                model=model,
                tokenizer=tokenizer)
pipe

In [ ]:
# 파이프라인을 이용한 생성
outputs = pipe(
    prompt,
    max_new_tokens=512,       # 생성할 수 있는 최대 새 토큰 수
    do_sample=True,           # 다음 토큰을 생성할 때 때 확률 분포에서 무작위로 샘플 여부
    temperature=0.5,          # 값이 낮을수록 높은 확률의 토큰을 선호, 높을수록 더 무작위 확률 선택
    top_k=64,                 # 확률이 높은 k개의 토큰 중 하나를 무작위로 선택
    top_p=0.95,               # 누적 확률이 p에 도달하는 토큰 집합에서 샘플링
    add_special_tokens=True
)
outputs

In [ ]:
# 결과 확인
print(outputs[0]["generated_text"])

In [ ]:
# 프롬프트 생성 함수
def gen_prompt(pipe, doc):
    messages = [
        {
            "role": "user",
            "content": "다음 문장은 영화리뷰입니다. 긍정 또는 부정으로 분류해주세요:\n\n{}".format(doc)
        }
    ]
    prompt = pipe.tokenizer.apply_chat_template(messages,
                                                tokenize=False,
                                                add_generation_prompt=True)
    return prompt

In [ ]:
# 문상 생성 함수
def gen_response(pipe, doc):
    prompt = gen_prompt(pipe, doc)

    outputs = pipe(
        prompt,
        max_new_tokens=512,        # 생성할 수 있는 최대 새 토큰 수
        do_sample=True,            # 다음 토큰을 생성할 때 때 확률 분포에서 무작위로 샘플 여부
        temperature=0.5,           # 값이 낮을수록 높은 확률의 토큰을 선호, 높을수록 더 무작위 확률 선택
        top_k=64,                  # 확률이 높은 k개의 토큰 중 하나를 무작위로 선택
        top_p=0.95,                # 누적 확률이 p에 도달하는 토큰 집합에서 샘플링
        add_special_tokens=True
    )
    return outputs[0]["generated_text"][len(prompt):]

In [ ]:
while True:
    doc = input('문장 > ')
    doc = doc.strip()
    if len(doc) == 0:
        break
    result = gen_response(pipe, doc)
    print(f'감정 > {result}\n\n')

## Gemma Training

### Train

In [ ]:
# !python train_llm_nsmc.py

In [ ]:
# 결과 확인
!ls -lh results/gemma-nsmc/lora_weight

In [ ]:
# 결과 확인
!ls -lh results/gemma-nsmc/merged

### Test

In [ ]:
# dataset
dataset = load_dataset("e9t/nsmc")

In [ ]:
# declare 4 bits quantize
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16
)
# load 4 bits model
model = AutoModelForCausalLM.from_pretrained("./results/gemma-nsmc/merged",
                                             attn_implementation='eager',
                                             device_map='auto',
                                             quantization_config=quantization_config)
# load tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.padding_side = 'right'

In [ ]:
# test를 위한 prompt를 생성 함수.
def gen_test_prompt(example):
    doc = example['document']
    prompt = r"""<bos><start_of_turn>user
다음 문장은 영화리뷰입니다. 긍정 또는 부정으로 분류해주세요:

{}<end_of_turn>
<start_of_turn>model
""".format(doc)
    return prompt

In [ ]:
# test prompt 생성 함수 동작 확인
for idx in range(10, 12):
    prompt = gen_test_prompt(dataset['test'][idx])
    print(prompt)
    print("*" * 50)

In [ ]:
doc = """엄청나게 재밌습니다. 강추!!!"""

messages = [
    {
        "role": "user",
        "content": "다음 문장은 영화리뷰입니다. 긍정 또는 부정으로 분류해주세요:\n\n{}".format(doc)
    }
]
prompt = tokenizer.apply_chat_template(messages,
                                       tokenize=False,
                                       add_generation_prompt=True)

In [ ]:
# pipeline 정의
pipe = pipeline("text-generation",
                model=model,
                tokenizer=tokenizer)

In [ ]:
# 파이프라인을 이용한 생성
outputs = pipe(
    prompt,
    max_new_tokens=512,       # 생성할 수 있는 최대 새 토큰 수
    do_sample=True,           # 다음 토큰을 생성할 때 때 확률 분포에서 무작위로 샘플 여부
    temperature=0.5,          # 값이 낮을수록 높은 확률의 토큰을 선호, 높을수록 더 무작위 확률 선택
    top_k=64,                 # 확률이 높은 k개의 토큰 중 하나를 무작위로 선택
    top_p=0.95,               # 누적 확률이 p에 도달하는 토큰 집합에서 샘플링
    add_special_tokens=True
)
print(outputs[0]["generated_text"])

In [ ]:
# 평가 (1000개만 평가)
total_sample_cnt, total_correct_cnt = 0, 0
for example in tqdm(dataset['test'].select(range(1000))):
    label = '긍정' if example['label'] == 1 else '부정'

    prompt = gen_test_prompt(example)
    outputs = pipe(
        prompt,
        max_new_tokens=512,        # 생성할 수 있는 최대 새 토큰 수
        do_sample=True,            # 다음 토큰을 생성할 때 때 확률 분포에서 무작위로 샘플 여부
        temperature=0.5,           # 값이 낮을수록 높은 확률의 토큰을 선호, 높을수록 더 무작위 확률 선택
        top_k=64,                  # 확률이 높은 k개의 토큰 중 하나를 무작위로 선택
        top_p=0.95,                # 누적 확률이 p에 도달하는 토큰 집합에서 샘플링
        add_special_tokens=True
    )
    pred = outputs[0]['generated_text'][len(prompt):]
    total_sample_cnt += 1
    total_correct_cnt += 1 if label == pred else 0
print(f"Test Accuracy: {total_correct_cnt} / {total_sample_cnt} = {total_correct_cnt/total_sample_cnt:.4f}")

### Infer

In [ ]:
# 프롬프트 생성 함수
def gen_prompt(pipe, doc):
    messages = [
        {
            "role": "user",
            "content": "다음 문장은 영화리뷰입니다. 긍정 또는 부정으로 분류해주세요:\n\n{}".format(doc)
        }
    ]
    prompt = pipe.tokenizer.apply_chat_template(messages,
                                                tokenize=False,
                                                add_generation_prompt=True)
    return prompt

In [ ]:
# 문상 생성 함수
def gen_response(pipe, doc):
    prompt = gen_prompt(pipe, doc)

    outputs = pipe(
        prompt,
        max_new_tokens=512,        # 생성할 수 있는 최대 새 토큰 수
        do_sample=True,            # 다음 토큰을 생성할 때 때 확률 분포에서 무작위로 샘플 여부
        temperature=0.5,           # 값이 낮을수록 높은 확률의 토큰을 선호, 높을수록 더 무작위 확률 선택
        top_k=64,                  # 확률이 높은 k개의 토큰 중 하나를 무작위로 선택
        top_p=0.95,                # 누적 확률이 p에 도달하는 토큰 집합에서 샘플링
        add_special_tokens=True
    )
    return outputs[0]["generated_text"][len(prompt):]

In [ ]:
while True:
    doc = input('문장 > ')
    doc = doc.strip()
    if len(doc) == 0:
        break
    result = gen_response(pipe, doc)
    print(f'감정 > {result}\n\n')

# RAG를 이용한 한국어 위키 검색 서비스

## 한국어 위키

### 한국어 위키 데이터 준비

In [ ]:
# 최신 wiki dump 다운로드
!wget https://dumps.wikimedia.org/kowiki/latest/kowiki-latest-pages-articles.xml.bz2 \
    -O ./data/kowiki-latest-pages-articles.xml.bz2

In [ ]:
# wiki dump 파일 전처리 (시간 오래 걸림)
# !sh ./extract_kowiki.sh

In [ ]:
# wiki 5개 페이지만 확인
wiki_list = []
with open("./data/kowiki/AA/wiki_00") as f:
    for i, line in enumerate(f):
        wiki = json.loads(line)
        print(wiki)
        wiki_list.append(wiki)
        if i == 4:
            break

In [ ]:
print(wiki_list[0]["text"])

In [ ]:
print(wiki_list[2]["text"])

### 한국어 위키 길이 분포 확인

In [ ]:
# 파일 목록 조회 (시간 관계상 100 파일만)
fn_list = glob.glob("./data/kowiki/*/wiki_*")
fn_list.sort()
fn_list = fn_list[:100]
len(fn_list)

In [ ]:
# 페이지 별 문장 길이 조사
len_list = []
for fn in tqdm(fn_list):
    with open(fn) as f:
        for line in f:
            line = line.strip()
            if line:
                data = json.loads(line)
                text_len = len(data["text"].split())
                len_list.append(text_len)
len(len_list)

In [ ]:
# 길이 분포 확인
min(len_list), max(len_list), sum(len_list) / len(len_list)

In [ ]:
# 길이 분포 확인
plt.hist(len_list, 10)
plt.show()

### 한국어 위키 페이지 분할 (chunk)

In [ ]:
# 페이지를 chunk 단위로 분할하는 함수 정의
def split_page_to_chunk(text, n_count=256):
    lines = text.split("\n")
    chunks, buf = [], []
    count = 0
    for line in lines:
        buf.append(line)
        count += len(line.split())
        if count >= n_count:
            chunks.append("\n".join(buf))
            buf.clear()
            buf.append(line)
            count = len(line.split())
    if len(buf) > 1:
        chunks.append("\n".join(buf))
        buf.clear()
    return chunks

In [ ]:
# 페이지를 chunk 단위로 분할
chunk_list = []
for fn in tqdm(fn_list):
    with open(fn) as f:
        for line in f:
            line = line.strip()
            if line:
                data = json.loads(line)
                chunks = split_page_to_chunk(data["text"])
                _id = int(data["id"])
                for i, chunk in enumerate(chunks):
                    chunk_list.append({
                        "id": f"{_id:06d}-{i:03d}",
                        "chunk": chunk,
                        "metadata": {
                            "docid": str(_id),
                            "chunkid": str(i),
                            "revid": data["revid"],
                            "title": data["title"],
                            "url": data["url"],
                        }
                    })
len(chunk_list)

In [ ]:
chunk_list[3]

In [ ]:
chunk_list[4]

## Chunk to Embedding

In [ ]:
# SentenceBERT 모델 생성
embed_model = SentenceTransformer('snunlp/KR-SBERT-V40K-klueNLI-augSTS')

In [ ]:
chunks = [data["chunk"] for data in chunk_list]
len(chunks)

In [ ]:
# Chunk Embedding 생성
chunk_embeddings = []
for i in trange(0, len(chunks), 128):
    embeddings = embed_model.encode(chunks[i:i+128], normalize_embeddings=True)
    chunk_embeddings.append(embeddings)
chunk_embeddings = np.concatenate(chunk_embeddings, axis=0)
chunk_embeddings.shape

## Vector Query

In [ ]:
# 검색 함수 정의
def query_sentence_transformer(embed_model, chunk_embeddings, query, top_n=5):
    query_embedding = embed_model.encode(query, normalize_embeddings=True)
    # score 계산
    doc_scores = np.dot(chunk_embeddings, query_embedding)
    # score 순서로 정렬
    rank = np.argsort(-doc_scores)
    # top-n
    query_result = []
    for i in rank[:top_n]:
        query_result.append((i, doc_scores[i]))
    return query_result

In [ ]:
# 대화형 검색
while True:
    query = input('검색 > ')
    query = query.strip()
    if len(query) == 0:
        break
    query_result = query_sentence_transformer(embed_model, chunk_embeddings, query)
    for i, score in query_result:
        print(f'---- score: {score} ----')
        print(chunk_list[i]["chunk"])
        print()

## Gemma-3 Loading

In [ ]:
# declare 4 bits quantize
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16
)
# load 4 bits model
model = AutoModelForCausalLM.from_pretrained(MODEL_ID,
                                             attn_implementation='eager',
                                             device_map='auto',
                                             quantization_config=quantization_config)
# load tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.padding_side = 'right'

In [ ]:
# 파이프라인 선언
pipe = pipeline("text-generation",
                model=model,
                tokenizer=tokenizer)
pipe

## Gemma-3 Q&A without Retriver

In [ ]:
# 프롬프트 생성 함수
def gen_prompt(pipe, query):
    messages = [
        {
            "role": "user",
            "content": f"""'질문'에 대해서 답변해 주세요.:

질문: {query}"""
        }
    ]
    prompt = pipe.tokenizer.apply_chat_template(messages,
                                                tokenize=False,
                                                add_generation_prompt=True)
    return prompt

In [ ]:
# 문상 생성 함수
def gen_response(pipe, query):
    prompt = gen_prompt(pipe, query)

    outputs = pipe(
        prompt,
        max_new_tokens=512,
        do_sample=True,
        temperature=0.5,
        top_k=64,
        top_p=0.95,
        add_special_tokens=True
    )
    return outputs[0]["generated_text"][len(prompt):]

In [ ]:
while True:
    query = input('질문 > ')
    query = query.strip()
    if len(query) == 0:
        break
    result = gen_response(pipe, query)
    print(f'답변 > {result}\n\n')

## Gemma-3 Q&A with Retriver

In [ ]:
# 검색 함수 정의
def query_sentence_transformer(embed_model, chunk_embeddings, query, top_n=5):
    query_embedding = embed_model.encode(query, normalize_embeddings=True)
    # score 계산
    doc_scores = np.dot(chunk_embeddings, query_embedding)
    # score 순서로 정렬
    rank = np.argsort(-doc_scores)
    # top-n
    query_result = []
    for i in rank[:top_n]:
        query_result.append((i, doc_scores[i]))
    return query_result

In [ ]:
# 프롬프트 생성 함수
def gen_prompt(pipe, query_docs, query):
    messages = [
        {
            "role": "user",
            "content": f"""당신이 가진 지식을 의존하지 말고 '문서1'부터 '문서5'를 참고해서 '질문'에 대해서 답변해 주세요.:

문서5: {query_docs[4]}

문서4: {query_docs[3]}

문서3: {query_docs[2]}

문서2: {query_docs[1]}

문서1: {query_docs[0]}

질문: {query}"""
        }
    ]
    prompt = pipe.tokenizer.apply_chat_template(messages,
                                                tokenize=False,
                                                add_generation_prompt=True)
    return prompt

In [ ]:
# 문상 생성 함수
def gen_response(pipe, query_docs, query):
    prompt = gen_prompt(pipe, query_docs, query)

    outputs = pipe(
        prompt,
        max_new_tokens=512,
        do_sample=True,
        temperature=0.5,
        top_k=64,
        top_p=0.95,
        add_special_tokens=True
    )
    return outputs[0]["generated_text"][len(prompt):]

In [ ]:
while True:
    query = input('질문 > ')
    query = query.strip()
    if len(query) == 0:
        break
    # 문서 검색
    query_result = query_sentence_transformer(embed_model, chunk_embeddings, query)
    query_docs = []
    for i, score in query_result:
        chunk = chunk_list[i]
        title = chunk['metadata']['title']
        document = chunk['chunk']
        query_docs.append(f"{title}\n{document}")

    result = gen_response(pipe, query_docs, query)
    print(f'답변 > {result}\n\n')